# HalDetect 2.5: DeBERTa-v3-large NLI Rescoring
This notebook replaces `facebook/bart-large-mnli` with `cross-encoder/nli-deberta-v3-large` as required by the Phase 6 remediation plan.

**Instructions:**
1. Turn on the **T4x2 GPU** accelerator in your Kaggle Notebook settings.
2. Upload `nli_rescore_worklist_both.json` via Add Input -> Upload Dataset.
3. Set `INPUT_FILE` to your dataset path.
4. Run all cells.
5. Download the generated `nli_rescore_scores_both_deberta.json` and place it in your local `current/` folder.

In [ ]:
!pip install transformers accelerate -q

In [ ]:
import json
import time
import glob
import os
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Automatic path detection or specify your exact path:
candidates = glob.glob("/kaggle/input/**/nli_rescore_worklist_both.json", recursive=True)
if candidates:
    INPUT_FILE = candidates[0]
else:
    INPUT_FILE = "/kaggle/input/datasets/sahibabchowdhury/nli-rescore-worklist-both/nli_rescore_worklist_both.json"

OUTPUT_FILE = "nli_rescore_scores_both_deberta.json"

print(f"Loading {INPUT_FILE}...")
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

meta = data["meta"]
items = data["items"]

# Extract premise-hypothesis pairs matching worklist schema
PAIRS = [(it["claim"], p["text"], p["fixed_key"]) for it in items for p in it.get("passages", [])]
print(f"Found {len(items)} rows, resolving to {len(PAIRS)} unique premise-hypothesis pairs.")

In [ ]:
model_name = "cross-encoder/nli-deberta-v3-large"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {model_name} on {device}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()
print("Model loaded successfully.")

In [ ]:
scores = {}
errors = []
print(f"Scoring {len(PAIRS)} pairs...")
t0 = time.time()

with torch.no_grad():
    for n, (claim, passage, key) in enumerate(PAIRS, 1):
        try:
            # NLI evaluates (premise, hypothesis) -> (passage, claim)
            inputs = tokenizer(passage, claim, truncation=True, max_length=512, return_tensors="pt").to(device)
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1).squeeze().cpu().tolist()
            
            # cross-encoder/nli-deberta-v3-large id2label:
            # 0: contradiction, 1: entailment, 2: neutral
            scores[key] = {
                "contradict": float(probs[0]),
                "entail": float(probs[1]),
                "neutral": float(probs[2])
            }
        except Exception as exc:
            errors.append((key, repr(exc)))

        if n % 10 == 0 or n == len(PAIRS):
            el = time.time() - t0
            print(f"  {n}/{len(PAIRS)}  {el:5.1f}s elapsed, ~{el / n * (len(PAIRS) - n):5.1f}s left")

print(f"\nScored {len(scores)} of {len(PAIRS)} pairs in {time.time() - t0:.1f}s")
if errors:
    print(f"WARNING: {len(errors)} errors occurred: {errors[:5]}")

In [ ]:
payload = {
    "meta": {
        "task": meta.get("task", "#16 nli__ cache-key collision rescore"),
        "set": meta.get("set", "both"),
        "model": model_name,
        "device": device,
        "pairs": len(scores),
        "rows": len(items)
    },
    "scores": scores
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=1)
    
print(f"\nSUCCESS: Output saved to {OUTPUT_FILE}.")
print("Please download this file from the Output tab and save it to your local current/ folder.")